<a href="https://colab.research.google.com/github/SURENAANERUS/Projects/blob/main/Copy_of_model19.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive') # import the data from google Drive

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import numpy as np
import torch
import gc

b = np.load('/content/drive/MyDrive/chess_dataset_3_2.npz') # load the data

'''
The plan:
1. Get the data in good order, verify that the model is shitty.
2. Collapse the data
3. Train it on collapsed data
4. Split the data by winner
5. Train the model only on black winner's moves.
'''
xNp = b['X'] # these are 18 matrices of inputs
yNp = b['y'] # these are 2 numbers, start position and end position
outcomes = b['outcomes']

In [12]:
# WARNING THIS TAKES AROUND 10 MINUTES!!!!
gc.collect()
torch.cuda.empty_cache()
use_cuda = torch.cuda.is_available()
device = "cuda" if use_cuda else "cpu"

x = torch.from_numpy(xNp).to("cuda")
y = torch.from_numpy(yNp).to("cuda")

x = x[:,:16]
print(x.shape, y.shape)

flat = x.reshape(x.shape[0], -1) # (6012255, 1024)
unique_flat, inverse = torch.unique(flat, dim=0, return_inverse=True)
unique_x = unique_flat.reshape(-1, 16, 8, 8).to("cuda")

print("devices: ", x.device, y.device, flat.device, unique_flat.device, unique_x.device,
      inverse.device)

# grouped_y[i] contains all y rows that mapped to unique_x[i]
grouped_y = [y[inverse == i] for i in range(unique_x.shape[0])]

print("All done!")

torch.Size([6012255, 16, 8, 8]) torch.Size([6012255, 2])
devices:  cuda:0 cuda:0 cuda:0 cuda:0 cuda:0 cuda:0
All done!


In [13]:
import torch.nn as nn

# now it's time to create the network itself.
# we start with B x 16 x 8 x 8
class Model(nn.Module):
  def __init__(self):
    super().__init__()
    self.reLu = nn.ReLU()
    self.bn0 = nn.BatchNorm2d(16)

    self.conv1 = nn.Conv2d(16,32,3,1,padding="same") # now its 32 x 8 x 8
    self.bn1 = nn.BatchNorm2d(32)

    self.conv2 = nn.Conv2d(32,64,3,1,padding="same") # now its  64
    self.bn2 = nn.BatchNorm2d(64)

    self.conv3 = nn.Conv2d(64,128,3,1,padding="same") # now its 128
    self.bn3 = nn.BatchNorm2d(128)

    self.ln = nn.Linear(8192,4096) # let's just do a linear layer for now.

  def forward(self, x: torch.Tensor) -> torch.Tensor:
    # x comes as B x 16 x 8 x 8
    #x = self.bn0(x)
    #print("enter 1 block")
    x = self.conv1(x)
    x = self.bn1(x)
    x = self.reLu(x)

    #print("enter 2 block")

    x = self.conv2(x)
    x = self.bn2(x)
    x = self.reLu(x)
    #print("enter 3 block")
    x = self.conv3(x)
    x = self.bn3(x)
    x = self.reLu(x) # 128 x 8 x 8



    x = x.view(-1,128 * 8 * 8) #
    #print("NEW SHAPE: ", x.shape)
    x = self.ln(x)
    x = x.squeeze(1)
    #print("final x shape: ", x.shape)
    return x

model = Model().to(device)
# now onto the train and test loops



In [14]:
# WARNING: THIS TAKES AROUND 10 MIN
import gc
import torch
import torch.nn.functional as F


gc.collect()
torch.cuda.empty_cache()

gc.collect()
torch.cuda.empty_cache()

use_cuda = torch.cuda.is_available()
device = "cuda" if use_cuda else "cpu"

# Build sparse labels from grouped_y
sparse_labels = []
for ys in grouped_y:
    class_ids = ys[:, 0] * 64 + ys[:, 1]
    counts = torch.bincount(class_ids, minlength=4096)
    nonzero = counts.nonzero().squeeze(1)
    probs = counts[nonzero].float()
    probs /= probs.sum()
    sparse_labels.append((nonzero, probs))


class SparseDataset(torch.utils.data.Dataset):
    def __init__(self, x, sparse_labels):
        self.x = x.float().to(device)
        self.sparse_labels = sparse_labels

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        ids, probs = self.sparse_labels[idx]
        dense = torch.zeros(4096).to(device)
        dense[ids] = probs
        return self.x[idx], dense


# Split
trainRatio = 0.7
trainLen = int(len(unique_x) * trainRatio)

x = unique_x.to(device)
xTrain, xTest = x[:trainLen], x[trainLen:]

trainData = SparseDataset(xTrain, sparse_labels[:trainLen])
testData = SparseDataset(xTest, sparse_labels[trainLen:])

print(f"Train: {len(trainData)}, Test: {len(testData)}")


Train: 3580323, Test: 1534425


In [15]:


def train_epoch(model, device, trainLoader, opt, epoch, log_interval):
    model.train()
    for batchIdx, (data, target) in enumerate(trainLoader):
        data, target = data.to(device), target.to(device)
        opt.zero_grad()
        output = model(data)
        loss = F.cross_entropy(output, target)
        loss.backward()
        opt.step()

        if batchIdx % log_interval == 0:
            print(f"Train Epoch: {epoch}, Batch: {batchIdx}, Loss: {loss.item():.6f}")


def test(model, device, testLoader, epoch):
    model.eval()
    testLoss = 0
    nCorrect = 0
    nTotal = 0
    batchNum = 0
    with torch.no_grad():
        for data, target in testLoader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            testLoss += F.cross_entropy(output, target).item()

            preds = output.argmax(dim=1)
            nCorrect += (target[torch.arange(len(preds), device=device), preds] > 0).sum().item()
            nTotal += len(target)
            batchNum += 1

    testLoss /= batchNum
    accuracy = nCorrect / nTotal
    print(f"Epoch {epoch} — Test loss: {testLoss:.4f}, Accuracy: {accuracy:.4f}")


In [16]:
bs = 4096 #
trainDL = torch.utils.data.DataLoader(trainData,bs)
testDL = torch.utils.data.DataLoader(testData,bs)

# Let it rip
EPOCHS = 30
optimizer = torch.optim.Adam(model.parameters()) # defcault lr is 0.01 IIRC

for epoch in range(EPOCHS):
  train_epoch(model,device,trainDL,optimizer,epoch,10000)
  print("train done!")
  test(model,device,testDL,epoch)
  print("test done! ")



Train Epoch: 0, Batch: 0, Loss: 8.467405
train done!
Epoch 0 — Test loss: 3.0781, Accuracy: 0.2244
test done! 
Train Epoch: 1, Batch: 0, Loss: 2.184306
train done!
Epoch 1 — Test loss: 2.8573, Accuracy: 0.2574
test done! 
Train Epoch: 2, Batch: 0, Loss: 1.760034
train done!
Epoch 2 — Test loss: 2.7747, Accuracy: 0.2707
test done! 
Train Epoch: 3, Batch: 0, Loss: 1.630660
train done!
Epoch 3 — Test loss: 2.7376, Accuracy: 0.2784
test done! 
Train Epoch: 4, Batch: 0, Loss: 1.549067


KeyboardInterrupt: 